In [0]:
from typing import Any

In [0]:
import logging
from py4j.java_gateway import JavaObject 
from pyspark.sql import SparkSession
from lms_internal import LMSInternal
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_sales_lms(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, lms_internal: LMSInternal
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ExposureSalesId String ,SourceSystem String ,PortfolioKey String ,PortfolioName String ,IngestionTimestamp timestamp ,"
        #     "BatchId String ,ExposureSalesKey String , SalesKey int ,CepalManagement boolean ,ExposureSalesCreatedDate timestamp ,ExposureSalesCreatedBy string ,"
        #     "Description string ,IsDeletedFlag boolean ,ExposureSalesLastUpdatedDate timestamp ,ExposureSalesLastUpdatedBy string ,"
        #     "ExposureSalesStatusCode int ,ExposureSalesStatus string ,ExposureKey string ,EquivalentAmount decimal(19,4) ,EquivalentAmountBy decimal(19,4) ,"
        #     "ExposureSalesDetailsKey int ,AssetOwnerPartyKey int ,AssetOwnerPartyCreatedDate timestamp ,AssetOwnerPartyCreatedUser string ,"
        #     "AssetOwnerPartyTaxRegistrationNumber string ,AssetOwnerPartyName string ,AssetOwnerPartyLastUpdatedDate timestamp ,"
        #     "AssetOwnerPartyLastUpdatedBy string, BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureSales"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "lms",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== loansales ==================

        loansales_extra_filters = None

        if loansales_extra_filters is not None:
            loansales_extra_filters.append(date_filter)
            loansales_final_filters = " AND ".join(loansales_extra_filters)
        else:
            loansales_final_filters = date_filter

        loansales_selected_fields = [
            "Id",
            "CepalManagement",
            "description",
            "isdeleted",
            "lastmodifieddate",
            "lastmodifieduser",
            "creationdate",
            "creationuser",
            "status",
            "status_description",
            "InvestorFK",
            "businessdate",
        ]

        loansales = lms_internal.loanSales()
        loansales = loansales.select(loansales_selected_fields)

        # ================== loansalesdetails ==================

        loansalesdetails_extra_filters = None

        if loansalesdetails_extra_filters is not None:
            loansalesdetails_extra_filters.append(date_filter)
            loansalesdetails_final_filters = " AND ".join(
                loansalesdetails_extra_filters
            )
        else:
            loansalesdetails_final_filters = date_filter

        loansalesdetails_selected_fields = [
            "Id",
            "LoanSalesFK",
            "AccountNumber",
            "EquivalentAmount",
            "EquivalentAmountbyuser",
            "businessdate",
        ]

        loansalesdetails = lms_internal.loanSalesDetails()
        loansalesdetails = loansalesdetails.select(loansalesdetails_selected_fields)

        # ================== investor ==================

        investor_extra_filters = None

        if investor_extra_filters is not None:
            investor_extra_filters.append(date_filter)
            investor_final_filters = " AND ".join(investor_extra_filters)
        else:
            investor_final_filters = date_filter

        investor_selected_fields = [
            "Id",
            "CreationDate",
            "CreationUser",
            "InvestorAFM",
            "InvestorName",
            "LastModifiedDate",
            "LastModifiedUser",
            "businessdate",
        ]

        investor = lms_internal.investor()
        investor = investor.select(investor_selected_fields)

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        loansales_loansalesdetails_condition = (
            loansales["Id"] == loansalesdetails["LoanSalesFK"]
        )

        loansales_investor_condition = loansales["InvestorFK"] == investor["Id"]

        loansalesdetails_portfolio_condition = (
            substring(loansalesdetails.AccountNumber, 1, 3)
            == portfolio.Prefix
        )

        target_table_lms = (
            loansales.join(
                loansalesdetails,
                loansales_loansalesdetails_condition,
                "inner",
            )
            .join(
                investor,
                loansales_investor_condition,
                "inner",
            )
            .join(
                portfolio,
                loansalesdetails_portfolio_condition,
                "left",
            )
        )

        exposure_sales_ExposureSaleId_primary_key_fields = [
            loansales.Id,
            loansalesdetails.AccountNumber,
            lit("lms"),
        ]

        target_table_lms = target_table_lms.withColumn(
            "ExposureSalesId",
            sha2(
                concat_ws(";", *exposure_sales_ExposureSaleId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "ExposureSalesKey",
            concat_ws(
                ";",
                loansales.Id,
                loansalesdetails.AccountNumber,
            ),
        )

        # ================== Selection of Final Columns ==================

        result_df_lms = target_table_lms.select(
            target_table_lms.ExposureSalesId.alias("ExposureSalesId"),
            lit("lms").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_lms.ExposureSalesKey.alias("ExposureSalesKey"),
            loansales.Id.alias("SalesKey"),
            loansales.CepalManagement.alias("CepalManagement"),
            loansales.creationdate.alias("ExposureSalesCreatedDate"),
            loansales.creationuser.alias("ExposureSalesCreatedBy"),
            loansales.description.alias("Description"),
            loansales.isdeleted.alias("IsDeletedFlag"),
            loansales.lastmodifieddate.alias("ExposureSalesLastUpdatedDate"),
            loansales.lastmodifieduser.alias("ExposureSalesLastUpdatedBy"),
            loansales.status.alias("ExposureSalesStatusCode"),
            loansales.status_description.alias("ExposureSalesStatus"),
            loansalesdetails.AccountNumber.alias("ExposureKey"),
            loansalesdetails.EquivalentAmount.alias("EquivalentAmount"),
            loansalesdetails.EquivalentAmountbyuser.alias("EquivalentAmountBy"),
            loansalesdetails.Id.alias("ExposureSalesDetailsKey"),
            investor.Id.alias("AssetOwnerPartyKey"),
            investor.CreationDate.alias("AssetOwnerPartyCreatedDate"),
            investor.CreationUser.alias("AssetOwnerPartyCreatedUser"),
            investor.InvestorAFM.alias("AssetOwnerPartyTaxRegistrationNumber"),
            investor.InvestorName.alias("AssetOwnerPartyName"),
            investor.LastModifiedDate.alias("AssetOwnerPartyLastUpdatedDate"),
            investor.LastModifiedUser.alias("AssetOwnerPartyLastUpdatedBy"),
            loansales.businessdate.alias("BusinessDate"),
        )

        result_df_lms.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
lms_internal = LMSInternal(spark, catalog_landed, run_date)

In [0]:
load_curated_exposure_sales_lms(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, lms_internal)